In [1]:
# ==========================================
# 🎙️ SPEECH EMOTION ANALYSIS SYSTEM
# Using Wav2Vec2 + Gradio
# ==========================================

!pip -q install transformers torch librosa soundfile gradio

import torch
import librosa
import gradio as gr
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

# Load pre-trained speech emotion model
MODEL_NAME = "superb/wav2vec2-base-superb-er"

extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
model = AutoModelForAudioClassification.from_pretrained(MODEL_NAME)

# Emotion labels
labels = model.config.id2label


# Emotion prediction function
def analyze_emotion(audio):
    if audio is None:
        return "Please upload or record an audio file."

    try:
        # Load audio
        speech, sample_rate = librosa.load(
            audio,
            sr=16000,
            mono=True
        )

        # Prepare audio for model
        inputs = extractor(
            speech,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True
        )

        # Predict emotion
        with torch.no_grad():
            outputs = model(**inputs)

        probabilities = torch.softmax(outputs.logits, dim=-1)[0]

        # Get highest probability emotion
        predicted_id = torch.argmax(probabilities).item()
        emotion = labels[predicted_id]
        confidence = probabilities[predicted_id].item() * 100

        # Show all emotion scores
        scores = ""
        for i, probability in enumerate(probabilities):
            scores += f"{labels[i]}: {probability.item()*100:.2f}%\n"

        return (
            f"🎭 Predicted Emotion: {emotion}\n"
            f"📊 Confidence: {confidence:.2f}%\n\n"
            f"Emotion Scores:\n{scores}"
        )

    except Exception as e:
        return f"Error: {str(e)}"


# ==========================================
# Gradio Interface
# ==========================================

with gr.Blocks(title="Speech Emotion Analysis") as app:

    gr.Markdown("""
    # 🎙️ Speech Emotion Analysis
    ### Analyze emotion from human speech using AI
    """)

    audio_input = gr.Audio(
        sources=["upload", "microphone"],
        type="filepath",
        label="🎤 Upload or Record Speech"
    )

    analyze_button = gr.Button("🔍 Analyze Emotion")

    result = gr.Textbox(
        label="🎭 Emotion Analysis Result",
        lines=10
    )

    analyze_button.click(
        fn=analyze_emotion,
        inputs=audio_input,
        outputs=result
    )

    gr.Markdown("""
    **Technology:** Python + Wav2Vec2 + PyTorch + Gradio

    **Note:** Emotion predictions are approximate AI classifications
    and should not be treated as a definitive assessment of a person's
    emotional state.
    """)


# Launch application
app.launch(share=True)

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.15k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  378MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/216 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

model.safetensors: downloading bytes:           |  0.00B            

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://29eb0bff4d8b4cdd73.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
